# PS- 3A: Credit Card Fraud Detection

Same four steps as every PS-3 notebook: normal data, encoder-decoder, training, threshold.


In [1]:
# Import all packages first
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from tensorflow.keras import models, layers


In [2]:
# 1. Load updated credit-card CSV. Class 0 is normal and Class 1 is fraud.
data = pd.read_csv("creditcard.csv")
x = data.drop("Class", axis=1)
y = data["Class"]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
normal = x_train[y_train == 0]
scaler = StandardScaler()
normal = scaler.fit_transform(normal)
x_test = scaler.transform(x_test)
x_train, x_val = train_test_split(normal, test_size=0.1, random_state=42)


In [3]:
# 2. Same autoencoder structure for every PS-3 dataset.
encoder = models.Sequential([
    layers.Input(shape=(x_train.shape[1],)),
    layers.Dense(20, activation="relu"),
    layers.Dense(14, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(4, activation="relu")
])
decoder = models.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(14, activation="relu"),
    layers.Dense(20, activation="relu"),
    layers.Dense(x_train.shape[1], activation="linear")
])
model = models.Sequential([encoder, decoder])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])


In [4]:
# 3. Train only on normal data.
model.fit(x_train, x_train, validation_data=(x_val, x_val),
          epochs=20, batch_size=512, shuffle=True, verbose=2)


Epoch 1/20
400/400 - 9s - 23ms/step - loss: 0.9161 - mae: 0.6433 - val_loss: 0.8109 - val_mae: 0.6104
Epoch 2/20
400/400 - 3s - 7ms/step - loss: 0.7487 - mae: 0.5853 - val_loss: 0.6912 - val_mae: 0.5739
Epoch 3/20
400/400 - 3s - 7ms/step - loss: 0.6793 - mae: 0.5667 - val_loss: 0.6379 - val_mae: 0.5580
Epoch 4/20
400/400 - 3s - 7ms/step - loss: 0.6318 - mae: 0.5509 - val_loss: 0.6004 - val_mae: 0.5436
Epoch 5/20
400/400 - 3s - 7ms/step - loss: 0.5967 - mae: 0.5384 - val_loss: 0.5740 - val_mae: 0.5332
Epoch 6/20
400/400 - 3s - 7ms/step - loss: 0.5755 - mae: 0.5296 - val_loss: 0.5594 - val_mae: 0.5264
Epoch 7/20
400/400 - 3s - 7ms/step - loss: 0.5627 - mae: 0.5236 - val_loss: 0.5621 - val_mae: 0.5274
Epoch 8/20
400/400 - 3s - 6ms/step - loss: 0.5575 - mae: 0.5209 - val_loss: 0.5439 - val_mae: 0.5179
Epoch 9/20
400/400 - 3s - 7ms/step - loss: 0.5493 - mae: 0.5171 - val_loss: 0.5416 - val_mae: 0.5159
Epoch 10/20
400/400 - 3s - 7ms/step - loss: 0.5443 - mae: 0.5147 - val_loss: 0.5355 - val_

In [5]:
# 4. High reconstruction error means fraud.
threshold = np.percentile(np.mean((x_val - model.predict(x_val, verbose=0)) ** 2, axis=1), 99)
error = np.mean((x_test - model.predict(x_test, verbose=0)) ** 2, axis=1)
prediction = (error > threshold).astype(int)
print("Threshold:", threshold)
print(classification_report(y_test, prediction, target_names=["Normal", "Fraud"], zero_division=0))

print("Number of anomalies detected:", prediction.sum())


Threshold: 2.4632480662114378
              precision    recall  f1-score   support

      Normal       1.00      0.99      0.99     56864
       Fraud       0.13      0.82      0.22        98

    accuracy                           0.99     56962
   macro avg       0.56      0.90      0.61     56962
weighted avg       1.00      0.99      0.99     56962

